In [1]:
import numpy as np
import os
import json
import pickle
import time
import datetime
import matplotlib.pyplot as plt
from tqdm import tqdm
from joblib import dump, load
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import cross_val_score, GridSearchCV, StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score, confusion_matrix
from sklearn.base import BaseEstimator, ClassifierMixin
from mne.decoding import CSP
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from scipy import signal

# Directory to save models
models_dir = '../models'
os.makedirs(models_dir, exist_ok=True)

# Setup logging
import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')
logger = logging.getLogger('eeg_pipeline')

class EnsembleClassifier(BaseEstimator, ClassifierMixin):
    """
    Custom Ensemble classifier that combines CSP and non-CSP models for improved performance
    """
    def __init__(self, standard_pipeline=None, csp_pipeline=None, voting='soft'):
        self.standard_pipeline = standard_pipeline
        self.csp_pipeline = csp_pipeline
        self.voting = voting
        self.is_fitted_ = False
        self.classes_ = None
        
    def fit(self, X, y, X_3d=None):
        if X_3d is None:
            # Try to automatically convert to 3D format for CSP
            try:
                n_samples = X.shape[0]
                n_channels = 64  # Assume 64 channels by default
                n_times = X.shape[1] // n_channels
                X_3d = X.reshape(n_samples, n_channels, n_times).astype(np.float64)
            except:
                raise ValueError("Could not convert to 3D format automatically for CSP. Please provide X_3d.")
        
        # Train standard pipeline
        if self.standard_pipeline is not None:
            self.standard_pipeline.fit(X, y)
        
        # Train CSP pipeline
        if self.csp_pipeline is not None and X_3d is not None:
            self.csp_pipeline.fit(X_3d, y)
        
        # Save classes
        self.classes_ = np.unique(y)
        self.is_fitted_ = True
        return self
        
    def predict_proba(self, X, X_3d=None):
        if not self.is_fitted_:
            raise ValueError("Model must be trained before prediction.")
        
        if X_3d is None:
            # Try to automatically convert to 3D format for CSP
            try:
                n_samples = X.shape[0]
                n_channels = 64  # Assume 64 channels by default
                n_times = X.shape[1] // n_channels
                X_3d = X.reshape(n_samples, n_channels, n_times).astype(np.float64)
            except:
                raise ValueError("Could not convert to 3D format automatically for CSP. Please provide X_3d.")
        
        # Calculate probabilities from both pipelines
        probs = []
        
        if self.standard_pipeline is not None:
            std_proba = self.standard_pipeline.predict_proba(X)
            probs.append(std_proba)
        
        if self.csp_pipeline is not None:
            csp_proba = self.csp_pipeline.predict_proba(X_3d)
            probs.append(csp_proba)
        
        # Average probabilities
        if len(probs) > 1:
            final_proba = np.mean(probs, axis=0)
        else:
            final_proba = probs[0]
        
        return final_proba
    
    def predict(self, X, X_3d=None):
        proba = self.predict_proba(X, X_3d)
        return self.classes_[np.argmax(proba, axis=1)]

def load_data(models_dir='../models'):
    """
    Load preprocessed EEG data from the specified directory
    """
    logger.info("Loading preprocessed data...")
    
    # Find the latest preprocessed files
    x_files = [f for f in os.listdir(models_dir) if f.startswith('X_preprocessed') and f.endswith('.npy')]
    y_files = [f for f in os.listdir(models_dir) if f.startswith('y_labels') and f.endswith('.npy')]
    
    if not x_files or not y_files:
        raise FileNotFoundError(f"Preprocessed data files not found in {models_dir}")
    
    # Sort by timestamp to get the latest
    x_files.sort(reverse=True)
    y_files.sort(reverse=True)
    
    X = np.load(os.path.join(models_dir, x_files[0]))
    y = np.load(os.path.join(models_dir, y_files[0]))
    
    logger.info(f"Loaded data: X shape {X.shape}, y shape {y.shape}")
    logger.info(f"Classes: {np.unique(y)}")
    
    # Try to load preprocessing info
    preprocessing_info = None
    info_files = [f for f in os.listdir(models_dir) if f.startswith('preprocessing_info') and f.endswith('.json')]
    if info_files:
        with open(os.path.join(models_dir, info_files[0]), 'r') as f:
            preprocessing_info = json.load(f)
    
    return X, y, preprocessing_info

def reshape_to_3d(X, n_channels=64):
    """
    Reshape the 2D EEG data to 3D format (epochs, channels, time)
    """
    n_samples = X.shape[0]
    n_times = X.shape[1] // n_channels
    return X.reshape(n_samples, n_channels, n_times).astype(np.float64)

def augment_data(X, y, X_3d=None, n_augmentations=1):
    """
    Augment the EEG data to increase dataset size and balance classes
    """
    logger.info("Augmenting data...")
    
    # Apply SMOTE for class balancing
    logger.info("Applying SMOTE for class balancing")
    smote = SMOTE(random_state=42)
    X_smote, y_smote = smote.fit_resample(X, y)
    
    # Create augmented datasets
    X_augmented = [X_smote]
    y_augmented = [y_smote]
    
    # Add Gaussian noise
    noise_level = 0.05
    logger.info(f"Adding Gaussian noise (level: {noise_level})")
    for i in range(n_augmentations):
        noise = np.random.normal(0, noise_level, X_smote.shape)
        X_augmented.append(X_smote + noise)
        y_augmented.append(y_smote)
    
    # Reshape 3D data if provided
    X_3d_final = None
    if X_3d is not None:
        try:
            # Get dimensions for 3D
            n_channels = X_3d.shape[1]
            n_times = X_3d.shape[2]
            n_samples_aug = len(y_smote)
            
            # Reshape SMOTE data
            X_smote_3d = X_smote.reshape(n_samples_aug, n_channels, n_times)
            X_3d_augmented = [X_smote_3d]
            
            # Add temporal shift
            shift_range = 5
            logger.info(f"Applying temporal shift (range: {shift_range})")
            for i in range(n_augmentations):
                X_shifted = np.zeros_like(X_smote_3d)
                shifts = np.random.randint(-shift_range, shift_range, size=n_samples_aug)
                
                for j, shift in enumerate(shifts):
                    if shift > 0:
                        X_shifted[j, :, shift:] = X_smote_3d[j, :, :-shift]
                        X_shifted[j, :, :shift] = X_smote_3d[j, :, :shift]
                    elif shift < 0:
                        X_shifted[j, :, :shift] = X_smote_3d[j, :, -shift:]
                        X_shifted[j, :, shift:] = X_smote_3d[j, :, shift:]
                    else:
                        X_shifted[j] = X_smote_3d[j]
                
                X_3d_augmented.append(X_shifted)
            
            # Combine all augmented 3D data
            X_3d_final = np.vstack(X_3d_augmented)
        except Exception as e:
            logger.warning(f"Error augmenting 3D data: {e}")
    
    # Combine all augmented data
    X_final = np.vstack(X_augmented)
    y_final = np.hstack(y_augmented)
    
    logger.info(f"Augmented data: X shape {X_final.shape}, y shape {y_final.shape}")
    logger.info(f"Class distribution: {np.bincount(y_final)}")
    
    return X_final, y_final, X_3d_final

def create_standard_pipelines():
    """
    Create a set of standard machine learning pipelines (non-CSP)
    """
    pipelines = {
        'PCA_SVM': Pipeline([
            ('scaler', StandardScaler()),
            ('dimension_reduction', PCA(n_components=0.95)),
            ('classifier', SVC(kernel='rbf', probability=True, class_weight='balanced'))
        ]),
        'PCA_RF': Pipeline([
            ('scaler', StandardScaler()),
            ('dimension_reduction', PCA(n_components=0.95)),
            ('classifier', RandomForestClassifier(n_estimators=100, class_weight='balanced'))
        ]),
        'PCA_MLP': Pipeline([
            ('scaler', StandardScaler()),
            ('dimension_reduction', PCA(n_components=0.95)),
            ('classifier', MLPClassifier(max_iter=1000, hidden_layer_sizes=(100, 50)))
        ]),
        'SMOTE_PCA_RF': ImbPipeline([
            ('sampling', SMOTE(random_state=42)),
            ('scaler', StandardScaler()),
            ('dimension_reduction', PCA(n_components=0.95)),
            ('classifier', RandomForestClassifier(n_estimators=100, class_weight='balanced'))
        ])
    }
    return pipelines

def create_csp_pipelines():
    """
    Create a set of CSP pipelines for motor imagery classification
    """
    pipelines = {
        'CSP_SVM': Pipeline([
            ('csp', CSP(n_components=8, reg=None, log=True, cov_est='epoch')),
            ('scaler', StandardScaler()),
            ('classifier', SVC(kernel='rbf', probability=True, class_weight='balanced'))
        ]),
        'CSP_RF': Pipeline([
            ('csp', CSP(n_components=8, reg=None, log=True, cov_est='epoch')),
            ('scaler', StandardScaler()),
            ('classifier', RandomForestClassifier(n_estimators=100, class_weight='balanced'))
        ]),
        'CSP_Shrinkage_SVM': Pipeline([
            ('csp', CSP(n_components=8, reg='shrinkage', log=True, cov_est='epoch')),
            ('scaler', StandardScaler()),
            ('classifier', SVC(kernel='rbf', probability=True, class_weight='balanced'))
        ])
    }
    return pipelines

def create_ensemble_model():
    """
    Create an ensemble model combining CSP and standard pipelines
    """
    # Standard pipeline
    standard_pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('pca', PCA(n_components=0.95)),
        ('classifier', RandomForestClassifier(n_estimators=100, class_weight='balanced'))
    ])
    
    # CSP pipeline
    csp_pipeline = Pipeline([
        ('csp', CSP(n_components=8, reg='ledoit_wolf', log=True, cov_est='epoch')),
        ('scaler', StandardScaler()),
        ('classifier', SVC(kernel='rbf', probability=True, class_weight='balanced'))
    ])
    
    # Create ensemble
    ensemble = EnsembleClassifier(
        standard_pipeline=standard_pipeline,
        csp_pipeline=csp_pipeline,
        voting='soft'
    )
    
    return ensemble

def create_param_grids():
    """
    Create parameter grids for hyperparameter tuning
    """
    standard_param_grids = {
        'PCA_SVM': {
            'dimension_reduction__n_components': [0.9, 0.95, 0.99],
            'classifier__C': [0.1, 1, 10],
            'classifier__gamma': ['scale', 'auto']
        },
        'PCA_RF': {
            'dimension_reduction__n_components': [0.9, 0.95, 0.99],
            'classifier__n_estimators': [100, 200],
            'classifier__max_depth': [None, 20]
        },
        'PCA_MLP': {
            'dimension_reduction__n_components': [0.9, 0.95, 0.99],
            'classifier__hidden_layer_sizes': [(50,), (100,), (100, 50)],
            'classifier__alpha': [0.0001, 0.001]
        },
        'SMOTE_PCA_RF': {
            'sampling__k_neighbors': [5],
            'dimension_reduction__n_components': [0.9, 0.95],
            'classifier__n_estimators': [100, 200],
            'classifier__max_depth': [None, 20]
        }
    }
    
    csp_param_grids = {
        'CSP_SVM': {
            'csp__n_components': [4, 6, 8],
            'csp__log': [True],
            'classifier__C': [0.1, 1, 10]
        },
        'CSP_RF': {
            'csp__n_components': [4, 6, 8],
            'csp__log': [True],
            'classifier__n_estimators': [100, 200],
            'classifier__max_depth': [None, 20]
        },
        'CSP_Shrinkage_SVM': {
            'csp__n_components': [4, 6, 8],
            'classifier__C': [0.1, 1, 10]
        }
    }
    
    return standard_param_grids, csp_param_grids

def evaluate_pipeline(pipeline, param_grid, X, y, X_3d=None, name='Pipeline', scoring='accuracy', cv=None):
    """
    Evaluate a pipeline with cross-validation and hyperparameter optimization
    """
    logger.info(f"Evaluating pipeline: {name}")
    
    if cv is None:
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    grid_search = GridSearchCV(
        pipeline,
        param_grid,
        cv=cv,
        scoring=scoring,
        n_jobs=-1,
        verbose=1,
        error_score='raise',
        return_train_score=True
    )
    
    try:
        # Determine if it's a CSP pipeline
        is_csp_pipeline = 'CSP' in name or hasattr(pipeline, 'named_steps') and 'csp' in pipeline.named_steps
        
        # Use appropriate data based on pipeline type
        if is_csp_pipeline and X_3d is not None:
            logger.info("Using 3D data format for CSP pipeline")
            grid_search.fit(X_3d.astype(np.float64), y)
        else:
            logger.info("Using 2D data format for standard pipeline")
            grid_search.fit(X, y)
        
        # Save results
        result = {
            'best_params': grid_search.best_params_,
            'best_score': grid_search.best_score_,
            'cv_results': grid_search.cv_results_,
            'best_estimator': grid_search.best_estimator_,
            'is_csp': is_csp_pipeline
        }
        
        logger.info(f"Best score for {name}: {result['best_score']:.3f} with {scoring}")
        logger.info(f"Best parameters: {result['best_params']}")
        return result
    except Exception as e:
        logger.error(f"Error evaluating {name}: {e}")
        return None

def evaluate_ensemble(ensemble, X, y, X_3d, name='Mixed_Ensemble', scoring='accuracy', cv=None):
    """
    Evaluate the mixed ensemble that combines CSP and non-CSP models
    """
    logger.info(f"Evaluating mixed ensemble: {name}")
    
    if cv is None:
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    try:
        # Train the full ensemble
        ensemble.fit(X, y, X_3d=X_3d)
        
        # Calculate score with cross-validation
        scores = []
        for fold_idx, (train_idx, test_idx) in enumerate(tqdm(cv.split(X, y), desc="Cross-validation")):
            X_train, X_test = X[train_idx], X[test_idx]
            y_train, y_test = y[train_idx], y[test_idx]
            
            # For CSP, we need 3D data
            if X_3d is not None:
                X_3d_train, X_3d_test = X_3d[train_idx], X_3d[test_idx]
            else:
                X_3d_train = X_3d_test = None
            
            # Train and predict
            ensemble.fit(X_train, y_train, X_3d=X_3d_train)
            y_pred = ensemble.predict(X_test, X_3d=X_3d_test)
            
            # Calculate score
            if scoring == 'accuracy':
                score = accuracy_score(y_test, y_pred)
            elif scoring == 'f1_weighted':
                score = f1_score(y_test, y_pred, average='weighted')
            elif scoring == 'f1_macro':
                score = f1_score(y_test, y_pred, average='macro')
            else:
                score = accuracy_score(y_test, y_pred)
            
            scores.append(score)
            logger.info(f"Fold {fold_idx+1}/{cv.get_n_splits()}: {score:.3f}")
        
        # Calculate average scores
        mean_score = np.mean(scores)
        std_score = np.std(scores)
        
        result = {
            'scores': scores,
            'mean_score': mean_score,
            'std_score': std_score,
            'best_estimator': ensemble,
            'is_csp': True  # It's a mixed model that includes CSP
        }
        
        logger.info(f"Average score for {name}: {mean_score:.3f} ± {std_score:.3f}")
        return result
    except Exception as e:
        logger.error(f"Error evaluating mixed ensemble {name}: {e}")
        return None

def evaluate_all_pipelines(X, y, X_3d, scoring='accuracy'):
    """
    Evaluate all pipelines and return the best one
    """
    logger.info(f"Starting comprehensive pipeline evaluation with {scoring} scoring...")
    
    # Create pipelines and parameter grids
    standard_pipelines = create_standard_pipelines()
    csp_pipelines = create_csp_pipelines()
    standard_param_grids, csp_param_grids = create_param_grids()
    
    results = {}
    best_score = 0
    best_pipeline = None
    best_pipeline_name = None
    
    # Create cross-validation object
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    # Evaluate standard pipelines
    for name, pipeline in standard_pipelines.items():
        result = evaluate_pipeline(
            pipeline,
            standard_param_grids.get(name, {}),
            X, y, None,
            name=name,
            scoring=scoring,
            cv=cv
        )
        if result is not None:
            results[name] = result
            if result['best_score'] > best_score:
                best_score = result['best_score']
                best_pipeline = result['best_estimator']
                best_pipeline_name = name
    
    # Evaluate CSP pipelines
    if X_3d is not None:
        for name, pipeline in csp_pipelines.items():
            result = evaluate_pipeline(
                pipeline,
                csp_param_grids.get(name, {}),
                X, y, X_3d,
                name=name,
                scoring=scoring,
                cv=cv
            )
            if result is not None:
                results[name] = result
                if result['best_score'] > best_score:
                    best_score = result['best_score']
                    best_pipeline = result['best_estimator']
                    best_pipeline_name = name
    
    # Evaluate Mixed Ensemble
    if X_3d is not None:
        mixed_ensemble = create_ensemble_model()
        result = evaluate_ensemble(
            mixed_ensemble,
            X, y, X_3d,
            name='Mixed_Ensemble',
            scoring=scoring,
            cv=cv
        )
        if result is not None:
            results['Mixed_Ensemble'] = result
            if result['mean_score'] > best_score:
                best_score = result['mean_score']
                best_pipeline = result['best_estimator']
                best_pipeline_name = 'Mixed_Ensemble'
    
    evaluation_result = {
        'results': results,
        'best_pipeline': best_pipeline,
        'best_pipeline_name': best_pipeline_name,
        'best_score': best_score,
        'scoring_metric': scoring
    }
    
    logger.info(f"Best pipeline: {best_pipeline_name} with score {best_score:.3f}")
    
    return evaluation_result

def evaluate_with_multiple_metrics(X, y, X_3d):
    """
    Evaluate pipelines using multiple metrics
    """
    # Define metrics to evaluate
    metrics = ['accuracy', 'f1_weighted', 'f1_macro']
    all_results = {}
    
    for metric in metrics:
        logger.info(f"\n\nEvaluating with metric: {metric.upper()}")
        evaluation_result = evaluate_all_pipelines(X, y, X_3d, scoring=metric)
        all_results[metric] = evaluation_result
        
        logger.info(f"Best pipeline for {metric}: {evaluation_result['best_pipeline_name']}")
        logger.info(f"Score: {evaluation_result['best_score']:.3f}")
    
    # Find the pipeline with best overall performance
    best_overall_score = 0
    best_overall_pipeline = None
    best_overall_name = None
    best_metric = None
    
    for metric, result in all_results.items():
        if result['best_score'] > best_overall_score:
            best_overall_score = result['best_score']
            best_overall_pipeline = result['best_pipeline']
            best_overall_name = result['best_pipeline_name']
            best_metric = metric
    
    logger.info(f"\nBest overall pipeline is {best_overall_name} with {best_metric} score of {best_overall_score:.3f}")
    
    return {
        'all_metric_results': all_results,
        'best_overall_pipeline': best_overall_pipeline,
        'best_overall_name': best_overall_name,
        'best_overall_score': best_overall_score,
        'best_metric': best_metric
    }

def save_results(evaluation_result, timestamp=None):
    """
    Save results and the best pipeline
    """
    if timestamp is None:
        timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
    
    # Create directory for this evaluation
    eval_dir = os.path.join(models_dir, f'evaluation_{timestamp}')
    os.makedirs(eval_dir, exist_ok=True)
    
    # Save all metric results if available
    if 'all_metric_results' in evaluation_result:
        all_results = evaluation_result['all_metric_results']
        
        for metric, metric_result in all_results.items():
            results_file = os.path.join(eval_dir, f'pipeline_evaluations_{metric}.json')
            
            with open(results_file, 'w') as f:
                serializable_results = {}
                for name, result in metric_result['results'].items():
                    if result is not None:
                        serializable_results[name] = {
                            'best_params': result.get('best_params', {}),
                            'best_score': float(result.get('best_score', result.get('mean_score', 0))),
                            'is_csp': result.get('is_csp', False)
                        }
                json.dump(serializable_results, f, indent=4)
    
    # Save best overall pipeline
    best_pipeline = evaluation_result['best_overall_pipeline']
    best_pipeline_name = evaluation_result['best_overall_name']
    best_score = evaluation_result['best_overall_score']
    best_metric = evaluation_result['best_metric']
    
    # Save using joblib
    best_pipeline_path = os.path.join(eval_dir, 'best_pipeline.joblib')
    dump(best_pipeline, best_pipeline_path)
    
    # Also save in main directory for easy access
    dump(best_pipeline, os.path.join(models_dir, 'best_pipeline.joblib'))
    
    # Save best pipeline info
    pipeline_info = {
        'best_pipeline_name': best_pipeline_name,
        'best_score': float(best_score),
        'best_metric': best_metric,
        'timestamp': timestamp,
        'pipeline_path': best_pipeline_path,
        'is_csp': best_pipeline_name.startswith('CSP') or 'Ensemble' in best_pipeline_name,
        'is_ensemble': 'Ensemble' in best_pipeline_name
    }
    
    with open(os.path.join(eval_dir, 'best_pipeline_info.json'), 'w') as f:
        json.dump(pipeline_info, f, indent=4)
    
    with open(os.path.join(models_dir, 'best_pipeline_info.json'), 'w') as f:
        json.dump(pipeline_info, f, indent=4)
    
    logger.info(f"Results saved in {eval_dir}")
    logger.info(f"Best pipeline saved as {os.path.join(models_dir, 'best_pipeline.joblib')}")
    
    return eval_dir

def visualize_confusion_matrix(pipeline, X, y, X_3d=None, title="Confusion Matrix"):
    """
    Visualize the confusion matrix for the model's predictions
    """
    # Determine if it's a CSP pipeline or ensemble
    is_csp = (hasattr(pipeline, 'named_steps') and 'csp' in pipeline.named_steps) or isinstance(pipeline, EnsembleClassifier)
    
    # Make predictions
    if isinstance(pipeline, EnsembleClassifier) and X_3d is not None:
        y_pred = pipeline.predict(X, X_3d)
    elif is_csp and X_3d is not None:
        y_pred = pipeline.predict(X_3d)
    else:
        y_pred = pipeline.predict(X)
    
    # Calculate confusion matrix
    cm = confusion_matrix(y, y_pred)
    
    # Plot
    plt.figure(figsize=(8, 6))
    plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
    plt.title(title)
    plt.colorbar()
    
    # Add labels
    classes = np.unique(y)
    tick_marks = np.arange(len(classes))
    plt.xticks(tick_marks, classes)
    plt.yticks(tick_marks, classes)
    
    # Add text
    thresh = cm.max() / 2.
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, format(cm[i, j], 'd'),
                     horizontalalignment="center",
                     color="white" if cm[i, j] > thresh else "black")
    
    plt.ylabel('True label')
    plt.xlabel('Predicted label')
    plt.tight_layout()
    
    return plt

def main():
    """
    Main function to run the pipeline optimization process
    """
    logger.info("Starting EEG Pipeline Optimization")
    
    # Load data
    try:
        X, y, preprocessing_info = load_data()
        
        # Get dimensions for 3D data
        if preprocessing_info and 'channels' in preprocessing_info:
            n_channels = len(preprocessing_info['channels'])
        else:
            n_channels = 64  # Default assumption
            
        # Reshape to 3D for CSP
        X_3d = reshape_to_3d(X, n_channels=n_channels)
        logger.info(f"Created 3D data for CSP with shape {X_3d.shape}")
        
        # Augment data (limited for speed)
        X_aug, y_aug, X_3d_aug = augment_data(X, y, X_3d, n_augmentations=1)
        
        # Evaluate with multiple metrics
        results = evaluate_with_multiple_metrics(X_aug, y_aug, X_3d_aug)
        
        # Save results
        timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
        eval_dir = save_results(results, timestamp)
        
        # Visualize results for best pipeline
        best_pipeline = results['best_overall_pipeline']
        best_name = results['best_overall_name']
        
        if best_pipeline is not None:
            # Plot confusion matrix
            plot = visualize_confusion_matrix(
                best_pipeline, X_aug, y_aug, X_3d_aug, 
                title=f"Confusion Matrix - {best_name}"
            )
            plot.savefig(os.path.join(eval_dir, 'confusion_matrix.png'))
            
            # Simulate real-time prediction
            logger.info("\nTesting real-time prediction simulation:")
            predictions, true_labels, times, metrics = simulate_real_time_prediction(
                best_pipeline,
                X_aug[:100],  # Using a subset for demonstration
                y_aug[:100],
                X_3d_aug[:100] if X_3d_aug is not None else None
            )
            
            # Save metrics
            with open(os.path.join(eval_dir, 'realtime_metrics.json'), 'w') as f:
                json.dump({k: float(v) for k, v in metrics.items()}, f, indent=4)
            
            logger.info(f"\nPipeline optimization complete! Best model: {best_name}")
            logger.info(f"All results saved to: {eval_dir}")
        else:
            logger.error("No optimal pipeline found")
    
    except Exception as e:
        logger.error(f"Error in pipeline optimization: {e}")
        raise e

def simulate_real_time_prediction(pipeline, X, y, X_3d=None, chunk_size=10):
    """
    Simulate real-time predictions to test model performance
    """
    logger.info("Simulating real-time prediction...")
    
    predictions = []
    true_labels = []
    times = []
    
    # Determine if it's a CSP or mixed pipeline
    is_csp = (hasattr(pipeline, 'named_steps') and 'csp' in pipeline.named_steps) or isinstance(pipeline, EnsembleClassifier)
    
    for i in tqdm(range(0, len(X), chunk_size), desc="Processing chunks"):
        X_chunk = X[i:i + chunk_size]
        y_chunk = y[i:i + chunk_size]
        
        # Prepare 3D data for CSP if needed
        X_3d_chunk = None
        if is_csp and X_3d is not None:
            X_3d_chunk = X_3d[i:i + chunk_size]
        
        start_time = time.time()
        
        # Make predictions based on pipeline type
        if isinstance(pipeline, EnsembleClassifier) and X_3d_chunk is not None:
            y_pred = pipeline.predict(X_chunk, X_3d_chunk)
        elif is_csp and X_3d_chunk is not None:
            y_pred = pipeline.predict(X_3d_chunk)
        else:
            y_pred = pipeline.predict(X_chunk)
        
        pred_time = time.time() - start_time
        times.append(pred_time)
        
        predictions.extend(y_pred.tolist())
        true_labels.extend(y_chunk.tolist())
        
        # Calculate current accuracy for feedback
        current_acc = accuracy_score(true_labels, predictions)
        if (i // chunk_size) % 5 == 0:  # Show every 5 chunks
            logger.info(f"Processed {i+len(X_chunk)}/{len(X)} samples. Current accuracy: {current_acc:.3f}")
    
    # Calculate final metrics
    final_metrics = {
        'accuracy': accuracy_score(true_labels, predictions),
        'f1_weighted': f1_score(true_labels, predictions, average='weighted'),
        'precision': precision_score(true_labels, predictions, average='weighted'),
        'recall': recall_score(true_labels, predictions, average='weighted'),
        'avg_prediction_time': np.mean(times),
        'max_prediction_time': np.max(times),
        'min_prediction_time': np.min(times)
    }
    
    logger.info("\nFinal metrics from simulation:")
    for metric_name, metric_value in final_metrics.items():
        logger.info(f" • {metric_name}: {metric_value:.4f}")
    
    return predictions, true_labels, times, final_metrics

if __name__ == "__main__":
    main()


2025-03-08 17:49:24,460 - eeg_pipeline - INFO - Starting EEG Pipeline Optimization
2025-03-08 17:49:24,460 - eeg_pipeline - INFO - Loading preprocessed data...
2025-03-08 17:49:24,520 - eeg_pipeline - INFO - Loaded data: X shape (90, 51264), y shape (90,)
2025-03-08 17:49:24,521 - eeg_pipeline - INFO - Classes: [2 3]
2025-03-08 17:49:24,551 - eeg_pipeline - INFO - Created 3D data for CSP with shape (90, 64, 801)
2025-03-08 17:49:24,551 - eeg_pipeline - INFO - Augmenting data...
2025-03-08 17:49:24,551 - eeg_pipeline - INFO - Applying SMOTE for class balancing
2025-03-08 17:49:24,790 - eeg_pipeline - INFO - Adding Gaussian noise (level: 0.05)
2025-03-08 17:49:24,955 - eeg_pipeline - INFO - Applying temporal shift (range: 5)
2025-03-08 17:49:25,072 - eeg_pipeline - INFO - Augmented data: X shape (184, 51264), y shape (184,)
2025-03-08 17:49:25,073 - eeg_pipeline - INFO - Class distribution: [ 0  0 92 92]
2025-03-08 17:49:25,082 - eeg_pipeline - INFO - 

Evaluating with metric: ACCURACY
2

Fitting 5 folds for each of 18 candidates, totalling 90 fits


2025-03-08 17:54:24,599 - eeg_pipeline - INFO - Best score for PCA_SVM: 0.956 with accuracy
2025-03-08 17:54:24,600 - eeg_pipeline - INFO - Best parameters: {'classifier__C': 10, 'classifier__gamma': 'scale', 'dimension_reduction__n_components': 0.95}
2025-03-08 17:54:24,600 - eeg_pipeline - INFO - Evaluating pipeline: PCA_RF
2025-03-08 17:54:24,601 - eeg_pipeline - INFO - Using 2D data format for standard pipeline


Fitting 5 folds for each of 12 candidates, totalling 60 fits
